In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for comprehensive testing of Product Sales Analysis logic updates in Databricks
# Purpose: Validate logic updates for bonus eligibility, performance flag, product performance band, and rep tier
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script tests the updated business logic for sales transactions and rep summary, including error handling, schema validation, data type checks, and output correctness. It covers unit, integration, and data quality tests for the specified logic, using PySpark DataFrames and asserts.

# from pyspark.sql import SparkSession  # SparkSession is already available in Databricks
from pyspark.sql import functions as F  
from pyspark.sql.types import (  
    StructType, StructField, IntegerType, StringType, FloatType, DoubleType, TimestampType, LongType
)
from pyspark.sql.utils import AnalysisException  

# -- Setup: Define schemas for test data
sales_transactions_schema = StructType([
    StructField("transaction_id", IntegerType(), False),
    StructField("rep_id", IntegerType(), False),
    StructField("sales_amount", StringType(), True),  # Use StringType to test type conversion
    StructField("bonus_eligibility", StringType(), True),
    StructField("performance_flag", StringType(), True),
    StructField("product_perf_band", StringType(), True),
    StructField("transaction_date", TimestampType(), True)
])

rep_summary_schema = StructType([
    StructField("rep_id", IntegerType(), False),
    StructField("total_rep_sales", StringType(), True),  # Use StringType to test type conversion
    StructField("rep_tier", StringType(), True),
    StructField("bonus_eligibility", StringType(), True)
])

final_output_schema = StructType([
    StructField("transaction_id", IntegerType(), False),
    StructField("rep_id", IntegerType(), False),
    StructField("sales_amount", StringType(), True),
    StructField("bonus_eligibility", StringType(), True),
    StructField("performance_flag", StringType(), True),
    StructField("product_perf_band", StringType(), True),
    StructField("transaction_date", TimestampType(), True),
    StructField("rep_tier", StringType(), True)
])

# -- Utility function: Read CSV file safely
def read_csv_safe(path, schema):
    """
    Safely reads a CSV file into a DataFrame with the given schema.
    Args:
        path (str): Path to the CSV file.
        schema (StructType): Schema for the DataFrame.
    Returns:
        DataFrame: Loaded DataFrame or None if file is missing/invalid.
    """
    try:
        df = spark.read.csv(path, header=True, schema=schema)
        return df
    except Exception as e:
        print(f"Error reading file {path}: {e}")
        return None

# -- Load test data from catalog volume files (no dbfs prefix)
sales_transactions_path = "/Volumes/purgo_databricks/purgo_playground/sales_transactions.csv"
rep_summary_path = "/Volumes/purgo_databricks/purgo_playground/rep_summary.csv"
final_output_path = "/Volumes/purgo_databricks/purgo_playground/final_output.csv"

sales_df = read_csv_safe(sales_transactions_path, sales_transactions_schema)
rep_summary_df = read_csv_safe(rep_summary_path, rep_summary_schema)
final_output_df = read_csv_safe(final_output_path, final_output_schema)

# -- Assert required columns exist
required_sales_cols = {"transaction_id", "rep_id", "sales_amount", "bonus_eligibility", "performance_flag", "product_perf_band", "transaction_date"}
required_rep_summary_cols = {"rep_id", "total_rep_sales", "rep_tier", "bonus_eligibility"}
required_final_output_cols = {"transaction_id", "rep_id", "sales_amount", "bonus_eligibility", "performance_flag", "product_perf_band", "transaction_date", "rep_tier"}

def assert_required_columns(df, required_cols, df_name):
    """
    Asserts that the required columns exist in the DataFrame.
    Args:
        df (DataFrame): DataFrame to check.
        required_cols (set): Set of required column names.
        df_name (str): Name of the DataFrame for error messages.
    Returns:
        None
    """
    if df is None:
        raise AssertionError(f"{df_name} dataset not found")
    missing = required_cols - set(df.columns)
    if missing:
        raise AssertionError(f"Missing required column(s) in {df_name}: {', '.join(missing)}")

assert_required_columns(sales_df, required_sales_cols, "sales_transactions")
assert_required_columns(rep_summary_df, required_rep_summary_cols, "rep_summary")
assert_required_columns(final_output_df, required_final_output_cols, "final_output")

# -- Data type validation for sales_amount and total_rep_sales
def assert_numeric_column(df, col_name, df_name):
    """
    Asserts that the specified column contains only numeric or null values.
    Args:
        df (DataFrame): DataFrame to check.
        col_name (str): Column name to validate.
        df_name (str): Name of the DataFrame for error messages.
    Returns:
        None
    """
    non_numeric = df.filter(
        (F.col(col_name).isNotNull()) & (~F.col(col_name).cast(DoubleType()).isNotNull())
    ).count()
    if non_numeric > 0:
        raise AssertionError(f"{col_name} must be numeric in {df_name}")

assert_numeric_column(sales_df, "sales_amount", "sales_transactions")
assert_numeric_column(rep_summary_df, "total_rep_sales", "rep_summary")

# -- Null value validation for sales_amount and total_rep_sales
def assert_no_null(df, col_name, df_name):
    """
    Asserts that the specified column does not contain null values.
    Args:
        df (DataFrame): DataFrame to check.
        col_name (str): Column name to validate.
        df_name (str): Name of the DataFrame for error messages.
    Returns:
        None
    """
    null_count = df.filter(F.col(col_name).isNull()).count()
    if null_count > 0:
        raise AssertionError(f"{col_name} cannot be null in {df_name}")

assert_no_null(sales_df, "sales_amount", "sales_transactions")
assert_no_null(rep_summary_df, "total_rep_sales", "rep_summary")

# -- Duplicate rep_id validation in rep_summary
def assert_no_duplicate_rep_id(df):
    """
    Asserts that there are no duplicate rep_id values in rep_summary.
    Args:
        df (DataFrame): rep_summary DataFrame.
    Returns:
        None
    """
    dup_count = df.groupBy("rep_id").count().filter(F.col("count") > 1).count()
    if dup_count > 0:
        raise AssertionError("Duplicate rep_id found in rep_summary")

assert_no_duplicate_rep_id(rep_summary_df)

# -- Logic update functions

def update_bonus_eligibility(df):
    """
    Updates bonus_eligibility to 'Yes' if sales_amount > 25000, else keeps existing logic.
    Args:
        df (DataFrame): sales_transactions DataFrame.
    Returns:
        DataFrame: Updated DataFrame with bonus_eligibility.
    """
    return df.withColumn(
        "bonus_eligibility",
        F.when(F.col("sales_amount").cast(DoubleType()) > 25000, F.lit("Yes"))
         .otherwise(F.col("bonus_eligibility"))
    )

def update_performance_flag(df):
    """
    Updates performance_flag based on sales_amount:
    - High: >9000
    - Medium: <=9000 and >7000
    - Low: <=7000
    Args:
        df (DataFrame): sales_transactions DataFrame.
    Returns:
        DataFrame: Updated DataFrame with performance_flag.
    """
    return df.withColumn(
        "performance_flag",
        F.when(F.col("sales_amount").cast(DoubleType()) > 9000, F.lit("High"))
         .when((F.col("sales_amount").cast(DoubleType()) <= 9000) & (F.col("sales_amount").cast(DoubleType()) > 7000), F.lit("Medium"))
         .otherwise(F.lit("Low"))
    )

def update_product_perf_band(df):
    """
    Updates product_perf_band based on sales_amount:
    - Excellent: >10000
    - Good: >8000
    - Moderate: >5000
    - Poor: <=5000 (retained for <=5000)
    Args:
        df (DataFrame): sales_transactions DataFrame.
    Returns:
        DataFrame: Updated DataFrame with product_perf_band.
    """
    return df.withColumn(
        "product_perf_band",
        F.when(F.col("sales_amount").cast(DoubleType()) > 10000, F.lit("Excellent"))
         .when(F.col("sales_amount").cast(DoubleType()) > 8000, F.lit("Good"))
         .when(F.col("sales_amount").cast(DoubleType()) > 5000, F.lit("Moderate"))
         .otherwise(F.lit("Poor"))
    )

def update_rep_tier(df):
    """
    Adds/updates rep_tier based on total_rep_sales:
    - Platinum Plus: >45000
    - Platinum: >35000
    - Gold: >25000
    - Silver: <=25000
    Args:
        df (DataFrame): rep_summary DataFrame.
    Returns:
        DataFrame: Updated DataFrame with rep_tier.
    """
    return df.withColumn(
        "rep_tier",
        F.when(F.col("total_rep_sales").cast(DoubleType()) > 45000, F.lit("Platinum Plus"))
         .when(F.col("total_rep_sales").cast(DoubleType()) > 35000, F.lit("Platinum"))
         .when(F.col("total_rep_sales").cast(DoubleType()) > 25000, F.lit("Gold"))
         .otherwise(F.lit("Silver"))
    )

# -- Apply logic updates to sales_transactions
sales_df_updated = update_bonus_eligibility(sales_df)
sales_df_updated = update_performance_flag(sales_df_updated)
sales_df_updated = update_product_perf_band(sales_df_updated)

# -- Apply logic updates to rep_summary
rep_summary_df_updated = update_rep_tier(rep_summary_df)

# -- Join rep_tier to sales transactions for final output
final_output_actual = sales_df_updated.join(
    rep_summary_df_updated.select("rep_id", "rep_tier"),
    on="rep_id",
    how="left"
)

# -- Schema validation: Ensure final output matches expected schema
def assert_schema_match(df, schema, df_name):
    """
    Asserts that the DataFrame schema matches the expected schema.
    Args:
        df (DataFrame): DataFrame to check.
        schema (StructType): Expected schema.
        df_name (str): Name of the DataFrame for error messages.
    Returns:
        None
    """
    actual_fields = [(f.name, f.dataType.typeName()) for f in df.schema.fields]
    expected_fields = [(f.name, f.dataType.typeName()) for f in schema.fields]
    if actual_fields != expected_fields:
        raise AssertionError(f"Schema mismatch in {df_name}: {actual_fields} != {expected_fields}")

assert_schema_match(final_output_actual, final_output_schema, "final_output_actual")

# -- Data quality validation: Allowed values for rep_tier, performance_flag, product_perf_band
def assert_allowed_values(df, col_name, allowed, df_name):
    """
    Asserts that all values in the column are within the allowed set.
    Args:
        df (DataFrame): DataFrame to check.
        col_name (str): Column name to validate.
        allowed (set): Set of allowed values.
        df_name (str): Name of the DataFrame for error messages.
    Returns:
        None
    """
    invalid = df.filter(~F.col(col_name).isin(list(allowed)) & F.col(col_name).isNotNull()).count()
    if invalid > 0:
        raise AssertionError(f"{col_name} contains invalid values in {df_name}")

assert_allowed_values(final_output_actual, "rep_tier", {"Platinum Plus", "Platinum", "Gold", "Silver", None}, "final_output_actual")
assert_allowed_values(final_output_actual, "performance_flag", {"High", "Medium", "Low", None}, "final_output_actual")
assert_allowed_values(final_output_actual, "product_perf_band", {"Excellent", "Good", "Moderate", "Poor", None}, "final_output_actual")

# -- Performance test: Ensure logic runs within reasonable time for batch
import time  
start_time = time.time()
_ = update_bonus_eligibility(sales_df)
_ = update_performance_flag(sales_df)
_ = update_product_perf_band(sales_df)
_ = update_rep_tier(rep_summary_df)
duration = time.time() - start_time
assert duration < 10, "Performance test failed: Logic took too long to execute"

# -- Integration test: Validate final output matches expected test data
def assert_final_output_matches(actual_df, expected_df):
    """
    Asserts that the actual final output matches the expected final output.
    Args:
        actual_df (DataFrame): Actual output DataFrame.
        expected_df (DataFrame): Expected output DataFrame.
    Returns:
        None
    """
    actual_rows = actual_df.select(sorted(final_output_schema.fieldNames())).orderBy("transaction_id").collect()
    expected_rows = expected_df.select(sorted(final_output_schema.fieldNames())).orderBy("transaction_id").collect()
    if actual_rows != expected_rows:
        raise AssertionError("Final output does not match expected output")

assert_final_output_matches(final_output_actual, final_output_df)

# -- Display the final output as required
display(final_output_actual)

# spark.stop()  # Do not stop SparkSession in Databricks
